In [69]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [26]:
data = pd.read_csv('diabetes.csv')

In [84]:
data

,Number of times pregnant,Plasma glucose concentration,Diastolic blood pressure,Triceps skin fold thickness,2-Hour serum insulin,Body mass index,Age,Class
0,6,148,72,35,0,33.6,50,positive
1,1,85,66,29,0,26.6,31,negative
2,8,183,64,0,0,23.3,32,positive
3,1,89,66,23,94,28.1,21,negative
4,0,137,40,35,168,43.1,33,positive
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,63,negative
764,2,122,70,27,0,36.8,27,negative
765,5,121,72,23,112,26.2,30,negative
766,1,126,60,0,0,30.1,47,positive


In [ ]:
data.columns

Index(['Number of times pregnant', 'Plasma glucose concentration',
       'Diastolic blood pressure', 'Triceps skin fold thickness',
       '2-Hour serum insulin', 'Body mass index', 'Age', 'Class'],
      dtype='str')

In [79]:
data.Class.value_counts()

Class
negative    500
positive    268
Name: count, dtype: int64

In [83]:
for i in data.columns:
    print(data[i].value_counts())

Number of times pregnant
1     135
0     111
2     103
3      75
4      68
5      57
6      50
7      45
8      38
9      28
10     24
11     11
13     10
12      9
14      2
15      1
17      1
Name: count, dtype: int64
Plasma glucose concentration
100    17
99     17
125    14
111    14
106    14
       ..
56      1
169     1
149     1
65      1
190     1
Name: count, Length: 136, dtype: int64
Diastolic blood pressure
70     57
74     52
78     45
68     45
72     44
64     43
80     40
76     39
60     37
0      35
62     34
66     30
82     30
88     25
84     23
90     22
58     21
86     21
50     13
56     12
54     11
52     11
92      8
75      8
65      7
94      6
85      6
48      5
96      4
44      4
110     3
98      3
100     3
106     3
30      2
108     2
55      2
104     2
46      2
40      1
122     1
95      1
102     1
61      1
24      1
38      1
114     1
Name: count, dtype: int64
Triceps skin fold thickness
0     227
32     31
30     27
27     23
23     22
33

In [35]:
data.isnull().any(axis=0) # There are no any null values

Number of times pregnant        False
Plasma glucose concentration    False
Diastolic blood pressure        False
Triceps skin fold thickness     False
2-Hour serum insulin            False
Body mass index                 False
Age                             False
Class                           False
dtype: bool

In [80]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Number of times pregnant      768 non-null    int64  
 1   Plasma glucose concentration  768 non-null    int64  
 2   Diastolic blood pressure      768 non-null    int64  
 3   Triceps skin fold thickness   768 non-null    int64  
 4   2-Hour serum insulin          768 non-null    int64  
 5   Body mass index               768 non-null    float64
 6   Age                           768 non-null    int64  
 7   Class                         768 non-null    str    
dtypes: float64(1), int64(6), str(1)
memory usage: 48.1 KB


In [43]:
x = data.iloc[:,:-1].values
y = list(data.iloc[:,-1].values)

In [45]:
len(y)

768

In [47]:
y_int = []
for s in y:
    if s == 'positive':
        y_int.append(1)
    else:
        y_int.append(0)

In [54]:
y = np.array(y_int, dtype='float64')

### Data Normalization

In [58]:
sc = StandardScaler()
x = sc.fit_transform(x)

In [ ]:
# Convert the data into tensors
x = torch.tensor(x)
y = torch.tensor(y).unsqueeze(1)

/tmp/ipykernel_11043/2675953426.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x)
/tmp/ipykernel_11043/2675953426.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y).unsqueeze(1)


### Creating and Loading Dataset

In [66]:
class Dataset(Dataset):

    def __init__(self,x,y):
        self.x = x
        self.y = y
    
    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)

In [67]:
dataset = Dataset(x,y)

In [70]:
train_loader = torch.utils.data.DataLoader(dataset=dataset,
                                           batch_size=32,
                                           shuffle=True)

### Building the Neural Network

In [74]:
class Model(nn.Module):

    def __init__(self,input_features, output_features):
        super(Model, self).__init__()
        self.fc1 = nn.Linear(input_features, 5)
        self.fc2 = nn.Linear(5, 4)
        self.fc3 = nn.Linear(4, 3)
        self.fc4 = nn.Linear(3, output_features)
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
    
    def forward(self,x):
        out = self.fc1(x)
        out = self.tanh(out)
        out = self.fc2(out)
        out = self.tanh(out)
        out = self.fc3(out)
        out = self.tanh(out)
        out = self.fc4(out)
        out = self.sigmoid(out)
        return out

In [75]:
net = Model(7,1)

criterion = torch.nn.BCELoss(size_average=True)

optimizer = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9)

/home/grow-lt-361/Code/Python/.venv/lib/python3.12/site-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


### Tranining Network

In [77]:
epochs = 200
for epoch in range(epochs):
    for inputs, labels in train_loader:
        inputs = inputs.float()
        labels = labels.float().squeeze()  # Remove extra dimension
        # Forward pass
        outputs = net(inputs).squeeze()  # Ensure output is [batch_size]
        # Loss Calculation
        loss = criterion(outputs, labels)
        # Clear the gradients buffer (w = w - l * gradient)
        optimizer.zero_grad()
        # Backward pass
        loss.backward()
        # Update the weights
        optimizer.step()
    
    # Accuracy Calculation
    output = (outputs > 0.5).float()
    accuracy = (output == labels).float().mean()
    # Print Statistics
    print("Epoch: {}/{} | Loss: {:.3f} | Accuracy: {:.3f}".format(epoch+1, epochs, loss, accuracy))

Epoch: 1/200 | Loss: 0.570 | Accuracy: 0.688
Epoch: 2/200 | Loss: 0.442 | Accuracy: 0.844
Epoch: 3/200 | Loss: 0.541 | Accuracy: 0.688
Epoch: 4/200 | Loss: 0.521 | Accuracy: 0.750
Epoch: 5/200 | Loss: 0.588 | Accuracy: 0.688
Epoch: 6/200 | Loss: 0.626 | Accuracy: 0.719
Epoch: 7/200 | Loss: 0.584 | Accuracy: 0.688
Epoch: 8/200 | Loss: 0.489 | Accuracy: 0.719
Epoch: 9/200 | Loss: 0.391 | Accuracy: 0.844
Epoch: 10/200 | Loss: 0.499 | Accuracy: 0.844
Epoch: 11/200 | Loss: 0.480 | Accuracy: 0.750
Epoch: 12/200 | Loss: 0.332 | Accuracy: 0.812
Epoch: 13/200 | Loss: 0.546 | Accuracy: 0.781
Epoch: 14/200 | Loss: 0.498 | Accuracy: 0.688
Epoch: 15/200 | Loss: 0.517 | Accuracy: 0.719
Epoch: 16/200 | Loss: 0.419 | Accuracy: 0.844
Epoch: 17/200 | Loss: 0.458 | Accuracy: 0.719
Epoch: 18/200 | Loss: 0.468 | Accuracy: 0.844
Epoch: 19/200 | Loss: 0.545 | Accuracy: 0.719
Epoch: 20/200 | Loss: 0.492 | Accuracy: 0.656
Epoch: 21/200 | Loss: 0.449 | Accuracy: 0.781
Epoch: 22/200 | Loss: 0.585 | Accuracy: 0.6